# 3. Data Preparation

Based on the issues and patterns identified during Data Understanding, this phase prepares the dataset for modeling by selecting variables, correcting quality issues, constructing new features, and applying preprocessing steps while avoiding data leakage.

In [1]:
# importing the libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# Loading the dataset and visualizing summary statistics
ds = pd.read_excel('../data/campaign.xlsx', engine='openpyxl')

## 3.1 Data Selection

In this step, the variables to be used in the modelling process were selected according to their relevance to the project objective: predicting, wether or not a customer will respond to a marketing campaign. The dataset was filtered to keep only relevant variables and exclude ones that do not contribute meaningfully or could compromise the validity of the model.

The following variables were removed:
- **Z_CostContact**
- **Z_Revenue**
- **ID**

The Z variables are constant and equal to 3 and 11 respectively and serve no purpose to our study. ID is also removed as it brings no predictive power, being only an identifier.


In [3]:
# working on a copy of the dataset
df = ds.copy()
df = df.drop(columns=['ID', 'Z_CostContact', 'Z_Revenue'])

## 3.2. Data Cleaning

 ### A. Missing Values

Missing values are handled after the train-test split so that values are estimated using only training data and there is no data leakage.

### B. Implausible values

During Data Understanding, several values were identified as implausible. Instead of removing all observations automatically, each issue was handled individually.

- Negative values in `MntFishProducts` were converted into missing values since spending cannot be negative, but the remaining customer information may still be useful.

- Unrealistic birth years were removed because values such as 1893, 1899, and 1900 imply customer ages above 120 years.

- For `MntSweetProducts` and `Income`, only very implausible values were treated as missing. Since the issue seemed isolated to these variables and the remaining customer characteristics looked reasonable, keeping the customer and correcting only the problematic value was considered more appropriate than removing the full record.

- Finally, `YOLO` and `Absurd` were removed from `Marital_Status` because they are extremely rare and non-meaningful categories. `Alone` was merged into `Single` since both describe essentially the same customer status.

In [4]:
# negative spending
df.loc[df['MntFishProducts'] < 0, 'MntFishProducts'] = np.nan

# unrealistic ages
df = df[df['Year_Birth'] >= 1930]

# implausible spending values
df.loc[df['MntSweetProducts'] > 10000,'MntSweetProducts'] = np.nan

# one extreme income value
df.loc[df['Income'] > 200000,'Income'] = np.nan

# rare/inconsistent labels
df = df[~df['Marital_Status'].isin(['YOLO','Absurd'])]

# merging categories
df['Marital_Status'] = (df['Marital_Status'].replace({'Alone':'Single'}))


df.reset_index(drop=True,inplace=True)

 ## 3.3. Construct Data

New variables were created to better represent customer behavior and improve interpretability. The goal was to transform existing information into more meaningful measures aligned with the project objective of predicting campaign response.

1. `Year_Birth` was replaced by `Age` since age is easier to interpret and more directly related to customer behavior.

2. The original enrollment date was converted into `Seniority`, representing how long the customer has been with the company, as customer behavior may vary depending on the length of the relationship.

3. Since spending across product categories showed similar patterns during the exploratory analysis, a `Total_Spending` variable was created to capture overall customer spending in a single measure.

4. A `Total_Purchases` variable was created to summarize purchasing activity across the different channels.

5. Previous campaign behavior showed a strong relationship with the target variable during EDA. For this reason, `Accepted_Total` was created to capture a customer's overall engagement with past campaigns.

In [5]:
# converting customer date to datetime
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])

# creating age
reference_year = df['Dt_Customer'].dt.year.max()
df['Age'] = reference_year - df['Year_Birth']

# creating seniority
reference_date = df['Dt_Customer'].max()
df['Seniority'] = (reference_date - df['Dt_Customer']).dt.days

# total spending
spending_cols = [
'MntWines',
'MntFruits',
'MntMeatProducts',
'MntFishProducts',
'MntSweetProducts',
'MntGoldProds'
]
df['Total_Spending'] = df[spending_cols].sum(axis=1, skipna=True)

# total purchases
purchase_cols = [
'NumDealsPurchases',
'NumWebPurchases',
'NumCatalogPurchases',
'NumStorePurchases'
]
df['Total_Purchases'] = df[purchase_cols].sum(axis=1, skipna=True)

# previous campaign engagement
campaign_cols = [
'AcceptedCmp1',
'AcceptedCmp2',
'AcceptedCmp3',
'AcceptedCmp4',
'AcceptedCmp5'
]
df['Accepted_Total'] = df[campaign_cols].sum(axis=1, skipna=True)


df[['Age', 'Seniority', 'Total_Spending', 'Total_Purchases', 'Accepted_Total']].head()
df = df.drop(columns=['Year_Birth', 'Dt_Customer'])

## 3.4 Format Data 

The data were reformatted to ensure compatibility with the selected modelling techniques. This included splitting the data, handling missing values, encoding categorical variables, scaling numerical features, and verifying that train and test sets shared the same structure.

### 3.4.1. Train-test Split

Since our data is not time-based, a random train-test split is appropriate. Since the target variable is imbalanced, stratified sampling was used to preserve class proportions across training and test sets.

In [6]:
X = df.drop(columns='Response')
y = df['Response']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

(1786, 28) (447, 28)


### 3.4.2. Missing Value Imputation

Imputation was performed after splitting so that replacement values were estimated only from the training data, avoiding data leakage.

In [7]:
cols_with_na = [
'Income',
'MntWines',
'MntFruits',
'MntGoldProds',
'MntFishProducts',
'MntSweetProducts'
]

medians = X_train[cols_with_na].median()

X_train[cols_with_na] = X_train[cols_with_na].fillna(medians)
X_test[cols_with_na] = X_test[cols_with_na].fillna(medians)

spending_cols = [
    'MntWines',
    'MntFruits',
    'MntMeatProducts',
    'MntFishProducts',
    'MntSweetProducts',
    'MntGoldProds'
]

# recreating total_spending after imputation
X_train['Total_Spending'] = X_train[spending_cols].sum(axis=1)

X_test['Total_Spending'] = X_test[spending_cols].sum(axis=1)

Since some spending variables contained missing values after cleaning, Total_Spending was recalculated after imputation to avoid underestimating customer expenditure.

 ### 3.4.3. Encoding Categorical Variables

One-hot encoding was applied to `Education` and `Marital_Status` so that the models treat each category as a separate group. `drop_first=True` was used to avoid the dummy variable trap by removing one category from each encoded variable, and `align()` was used to ensure train and test sets have the same columns.

In [8]:
categorical_cols = ['Education','Marital_Status']

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True,dtype=int)

X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True, dtype=int)

# aligning the train and test sets
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

 ### 3.4.4. Assessing Multicollinearity

 The purpose of this step was to identify and address multicollinearity among the predictor variables since high multicollinearity can negatively impact model interpretability (particularly linear ones). To assess this, the Variance Inflation Factor (VIF) was calculated for the numerical features. It is performed using only the training data to avoid incorporating information from the test set.

 ###  Running  VIF

In [9]:
vif_cols = [col for col in X_train.columns if X_train[col].nunique()>2]

def calculate_vif(X):
    X = sm.add_constant(X)
    vif_data = pd.DataFrame()
    vif_data['Feature'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data.drop(index=0)

vif_results = calculate_vif(X_train[vif_cols])

vif_results.sort_values(by='VIF', ascending=False)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Feature,VIF
11,NumDealsPurchases,inf
13,NumCatalogPurchases,inf
19,Total_Purchases,inf
18,Total_Spending,inf
5,MntWines,inf
6,MntFruits,inf
7,MntMeatProducts,inf
8,MntFishProducts,inf
9,MntSweetProducts,inf
10,MntGoldProds,inf


#### Key Findings: 

VIF analysis showed high multicollinearity among some engineered variables, especially aggregate variables such as `Total_Spending` and `Total_Purchases` and their individual components. This was expected, since these variables were created from related customer activity measures. The aggregated variables capture overall customer behavior, while the original variables keep more detailed information. For example, two customers may have the same total number of purchases but very different purchasing habits across channels.
Since the final selected model was tree-based, the variables were retained to preserve potentially useful information for prediction.

### 3.4.5. Scaling data

In [10]:
# columns to scale (continuous, not binary)
scale_cols = ['Income','Recency','MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds','Age','Seniority','Total_Spending','Total_Purchases', 'Accepted_Total']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])

X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print(X_train.columns.equals(X_test.columns))

print(X_train_scaled.columns.equals(X_test_scaled.columns))

True
True


 ### Saving the datasets

In [11]:
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)